# Занятие 8, пара 1. Визуализация. Выбор графика


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## График начинается с вопроса

В прошлых тетрадях мы научились получать таблицу: очищать, объединять, группировать и запрашивать её через SQL.
Теперь хотим объяснить результат человеку. Можно нарисовать красивую картинку и всё равно ответить не на тот вопрос.
Сегодня будем замечать такие случаи.

Нужны Pandas и Matplotlib, локальные данные уже в комплекте. После пары вы сможете выбрать между
столбцами, гистограммой и диаграммой рассеяния, объяснить единицу наблюдения и подписать график так,
чтобы его можно было понять без вашего устного рассказа. Итог — три осмысленных графика и разбор одной ловушки.
Python и группировки не изучаем заново; новые команды построения разбираем до применения.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Сначала прочитаем таблицу

Работаем с **Palmer Penguins**: реальные измерения пингвинов трёх видов, собранные Kristen Gorman
и Palmer Station LTER. Пакет подготовили Allison Horst, Alison Hill и Kristen Gorman.
Это локальная копия открытого набора CC0, не данные нашей группы. Источник и контрольная сумма —
`course_support/data/visualization/SOURCE.md`; [описание набора](https://allisonhorst.github.io/palmerpenguins/).

Одна строка — одно наблюдение птицы. `species` — вид, `island` — остров, `body_mass_g` — масса в граммах,
`flipper_length_mm` — длина ласта в миллиметрах, `year` — год наблюдения.
Есть и размеры клюва, и поле `sex`. Пропуск измерения не равен нулевой массе: нулевой пингвин нам не нужен.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

penguins = pd.read_csv(DATA_DIR / 'visualization' / 'penguins.csv')
display(penguins.head())
print('Наблюдений:', len(penguins))
print(penguins.isna().sum())


### 2. Один маленький график

`plt.subplots()` создаёт **фигуру** `fig` и **область рисования** `ax`. Пока область одна.
Через `ax.bar(категории, высоты)` рисуем столбцы. Это удобно, когда вопрос звучит «сколько в каждой группе?».
Сначала отделим построение от оформления: увидим, чего графику не хватает.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['A', 'B', 'C'], [4, 7, 3])
plt.show()


Столбцы есть. А что мы посчитали? Карандаши, птиц, ошибки? Без подписей приходится гадать.
`set_title` задаёт заголовок, `set_xlabel` и `set_ylabel` — подписи осей.
`figsize=(7, 4)` — размер фигуры в дюймах. Это размер рисунка, а не диапазон значений.
Теперь возьмём настоящие данные. Знакомая `groupby(...).size()` считает **строки**, включая строки с пропуском массы.
`sort_values()` сортирует результат по количеству. Индекс Series хранит названия групп.


In [ ]:
species_count = penguins.groupby('species').size()
species_count = species_count.sort_values()
print(species_count)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(species_count.index, species_count.values)
ax.set_title('Сколько наблюдений каждого вида в наборе Palmer Penguins')
ax.set_xlabel('Вид')
ax.set_ylabel('Наблюдений, шт.')
plt.show()


Мы сравнили **представленность в этом наборе**, а не численность видов на Земле.
Такой вывод требует другой процедуры сбора данных. Большой столбец не делает вид «успешнее» —
возможно, исследователи просто чаще работали там, где он живёт.

Столбцы начинаются с нуля: длина столбца кодирует величину. Для длинных названий можно использовать
`ax.barh(названия, значения)` — горизонтальный вариант с тем же смыслом.


### ✏️ Ваш ход — Знаменатель меняет ответ

Две экспедиции проверяли измерения. Ниже **вымышленные** итоги:

| Экспедиция | Всего измерений | Измерений с ошибкой |
|---|---:|---:|
| A | 100 | 12 |
| B | 20 | 5 |

Кто чаще ошибался **в расчёте на измерение**? Создайте список `error_rates` из двух долей (0…1),
нарисуйте подходящий график и сформулируйте вывод. Дополнительно объясните, почему исходное количество ошибок
даёт другой рейтинг. Все нужные операции — деление, список, bar и подписи — уже разобраны.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.1', error_rates)


**Мой вывод:** …


### 3. Как распределена масса?

Количество видов мы уже сравнили. Теперь вопрос другой: «какие массы встречаются часто, а какие редко?».
Столбец на каждую птицу превратит график в частокол. Объединим близкие массы в **интервалы**.

Для этого вопроса нужны измерения массы. `dropna()` убирает пропуски только из выбранной Series;
строки исходной таблицы мы не удаляем. Посчитаем один интервал вручную: от 3000 включительно до 3500 не включительно.


In [ ]:
mass = penguins['body_mass_g'].dropna()
inside = (mass >= 3000) & (mass < 3500)
print('Измерений массы:', len(mass))
print('В интервале [3000, 3500):', inside.sum())


**Гистограмма** повторяет этот подсчёт для нескольких интервалов.
`ax.hist(mass, bins=edges)` принимает исходные измерения и границы интервалов, а не уже готовые высоты.
Все интервалы, кроме последнего, не включают правую границу; последний включает её.
Здесь ширина каждого интервала 500 г, поэтому высоты можно прямо сравнивать как количества.
Это не bar по отдельным категориям: соседство числовых интервалов имеет смысл.


In [ ]:
edges = [2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(mass, bins=edges, edgecolor='white')
ax.set_title('Распределение измеренной массы')
ax.set_xlabel('Масса, г')
ax.set_ylabel('Наблюдений в интервале')
plt.show()


Границы интервалов — часть решения аналитика. Слишком широкие скрывают детали, слишком узкие
подчёркивают случайные колебания. По одному красивому «горбу» нельзя объявить закон природы.

Чтобы сравнить два графика, создадим две области: `plt.subplots(1, 2)` означает одну строку и два столбца.
`axes[0]` — первая область, `axes[1]` — вторая. `sharex=True, sharey=True` оставляет общий масштаб.
`fig.tight_layout()` помогает подписям не налезать друг на друга.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
axes[0].hist(mass, bins=4, edgecolor='white')
axes[1].hist(mass, bins=24, edgecolor='white')
axes[0].set_title('4 интервала')
axes[1].set_title('24 интервала')
for ax in axes:
    ax.set_xlabel('Масса, г')
    ax.set_ylabel('Наблюдений')
fig.tight_layout()
plt.show()


### ✏️ Ваш ход — Чужой график не выдержал сравнения

Коллега сравнил два набора и объявил: «B сильнее сосредоточен около середины».
Он построил A с границами `[0, 2, 4, 6, 8]`, а B — с `[0, 8]`.

Данные: `sample_a = [1, 2, 3, 5, 6, 7]`, `sample_b = [1, 2, 3, 5, 6, 7]`.
Проверьте утверждение: выберите общие границы `common_edges`, постройте два сравнимых графика,
напишите, откуда взялось различие. Границы должны покрыть все данные и дать хотя бы два интервала.
Не пытайтесь «улучшить» данные, исправьте способ сравнения.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.2', common_edges)


**Мой вывод:** …


### 4. Длинный ласт — тяжёлая птица?

Здесь у каждой птицы есть **два измерения**. `ax.scatter(x, y)` рисует по точке на пару значений.
Координата по горизонтали — ласт, по вертикали — масса. Соединять точки линиями не нужно:
порядок строк в CSV не задаёт путь движения птицы.

В `dropna(subset=[...])` перечисляем только поля, необходимые этому вопросу. Мы не выбрасываем
птицу из-за пропуска пола, когда пол вообще не используется.


In [ ]:
pairs = penguins.dropna(subset=['flipper_length_mm', 'body_mass_g'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pairs['flipper_length_mm'], pairs['body_mass_g'], alpha=0.6)
ax.set_xlabel('Длина ласта, мм')
ax.set_ylabel('Масса, г')
ax.set_title('Одна точка — одна птица с двумя измерениями')
plt.show()


`alpha=0.6` делает точки полупрозрачными: наложения заметнее. Общий наклон есть,
но мы смешали виды. Посмотрим, сохранится ли впечатление после разделения.
Цикл по `groupby('species')` даёт имя вида и его таблицу. `label` присваивает имя серии,
а `legend()` показывает эти имена на графике. Это тот же scatter, только три раза.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for species, part in pairs.groupby('species'):
    ax.scatter(part['flipper_length_mm'], part['body_mass_g'], label=species, alpha=0.6)
ax.set_xlabel('Длина ласта, мм')
ax.set_ylabel('Масса, г')
ax.set_title('Связь измерений с учётом вида')
ax.legend()
plt.show()


Теперь видно, что общий рисунок складывается и из различий **между** видами, и из различий
**внутри** вида. Это повод уточнить анализ, а не объявить всю связь ошибкой.
Наблюдение связи не доказывает, что увеличение ласта само по себе увеличивает массу.
В следующей тетради превратим такую оговорку в проверяемый срез отчёта.

### 5. Масштаб тоже рассказывает историю

`ax.set_ylim(нижняя, верхняя)` задаёт видимый диапазон оси Y. Обрезанная ось у столбцов может
сильно преувеличить различие длин. Для точек или линии узкий диапазон иногда полезен,
но границы должны быть видны и соответствовать вопросу.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['До', 'После'], [98, 100])
ax.set_ylim(97, 101)
ax.set_ylabel('Успешных операций из 100')
ax.set_title('Учебный пример: похоже на огромный рост?')
plt.show()


### ✏️ Ваш ход — Починить сообщение

Вымышленный рекламный слайд выше сопровождается текстом «Качество выросло в три раза».
Постройте честное сравнение тех же чисел. Запишите прирост доли успехов в **процентных пунктах**
в `gain_pp`, а отношение нового количества к старому — в `growth_ratio`.
Сформулируйте подпись, которая не путает эти величины. Вы вправе оставить столбцы или выбрать точки,
но объясните масштаб. Не меняйте исходные наблюдения.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.1.3', gain_pp, growth_ratio)


**Мой вывод:** …


## Что сохраняем

В своей копии оставьте графики, ответы и выводы. Если нужен отдельный рисунок, у фигуры есть
`fig.savefig(путь, dpi=150, bbox_inches='tight')`: dpi задаёт разрешение, tight учитывает подписи.
Сначала создайте папку, потом сохраняйте файл. Следующая ячейка сохраняет отдельный основной пример,
поэтому не зависит от выполнения упражнений.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'visualization'
output_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(species_count.index, species_count.values)
ax.set_title('Состав набора Palmer Penguins')
ax.set_ylabel('Наблюдений, шт.')
fig.savefig(output_dir / 'species_count.png', dpi=150, bbox_inches='tight')
plt.show()
print(output_dir / 'species_count.png')


## Проверим себя

- «Сколько по категориям?» → столбцы. «Как распределено число?» → гистограмма.
- «Как связаны два измерения?» → scatter. Для изменения во времени нужна упорядоченная временная ось.
- На каждом графике понятны наблюдение, единицы, масштаб и границы вывода.

**Дополнительно на 10–15 минут.** Посчитайте наблюдения по годам, отсортируйте индекс через `sort_index()`.
Постройте линию `ax.plot(годы, количества, marker='o')` и объясните, почему она не показывает рост
популяции пингвинов. Затем сравните по годам доли видов. Изменится ли интерпретация?
Это отдельное исследование, основная работа от него не зависит.

Официальные примеры: [Matplotlib — гистограммы](https://matplotlib.org/stable/gallery/statistics/hist.html).
По желанию: Коул Нафлик, «Сторителлинг на языке данных» — читать с вопросом «какое решение должен
принять зритель?», а не собирать коллекцию декоративных приёмов.
